<a href="https://colab.research.google.com/github/localhost8800/D/blob/main/5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1) Generate an image by using Restricted Boltzmann Machine using a real
# image dataset (Use MNIST dataset). Use the DataLoader class of the
# torch.utils.data library to load training and testing datasets , set the batch size
# to 64 and apply the transformations.Visualize the real and generated image.

# Assignment 5 - Set A - Q1
# Image Generation using RBM and MNIST Dataset

import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# Transformations
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Lambda(lambda x: (x > 0.5).float())
])

# Load MNIST dataset
train_data = datasets.MNIST(
    root="./data", train=True, download=True, transform=transform
)

test_data = datasets.MNIST(
    root="./data", train=False, download=True, transform=transform
)

# Create DataLoaders
train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = DataLoader(test_data, batch_size=64, shuffle=False)

# Define RBM
class RBM(nn.Module):
    def __init__(self, n_visible=784, n_hidden=128):
        super().__init__()
        self.W = nn.Parameter(torch.randn(n_hidden, n_visible) * 0.01)
        self.v_bias = nn.Parameter(torch.zeros(n_visible))
        self.h_bias = nn.Parameter(torch.zeros(n_hidden))

    def sample_h(self, v):
        prob = torch.sigmoid(v @ self.W.t() + self.h_bias)
        return prob, torch.bernoulli(prob)

    def sample_v(self, h):
        prob = torch.sigmoid(h @ self.W + self.v_bias)
        return prob, torch.bernoulli(prob)

    def forward(self, v):
        _, h = self.sample_h(v)
        _, v = self.sample_v(h)
        return v

# Initialize RBM
rbm = RBM()
optimizer = torch.optim.SGD(rbm.parameters(), lr=0.1)

# Train RBM using Contrastive Divergence
epochs = 5

for epoch in range(epochs):
    total_loss = 0

    for images, _ in train_loader:
        images = images.view(-1, 784)

        # Positive phase
        _, h = rbm.sample_h(images)

        # Negative phase
        _, v_reconstructed = rbm.sample_v(h)
        _, h_reconstructed = rbm.sample_h(v_reconstructed)

        # Update weights and biases
        optimizer.zero_grad()

        loss = torch.mean((images - v_reconstructed) ** 2)
        loss.backward() if loss.requires_grad else None

        with torch.no_grad():
            rbm.W.grad = (
                h.t() @ images - h_reconstructed.t() @ v_reconstructed
            ) / images.size(0)
            rbm.v_bias.grad = torch.mean(images - v_reconstructed, dim=0)
            rbm.h_bias.grad = torch.mean(h - h_reconstructed, dim=0)

        optimizer.step()
        total_loss += loss.item()

    print("Epoch:", epoch + 1, "Loss:", total_loss / len(train_loader))

# Get real images
real_images, _ = next(iter(test_loader))

# Generate images from random hidden states
with torch.no_grad():
    hidden = torch.bernoulli(torch.rand(16, 128))
    generated, _ = rbm.sample_v(hidden)

# Visualize real and generated images
fig, axes = plt.subplots(2, 8, figsize=(12, 4))

for i in range(8):
    axes[0, i].imshow(real_images[i].squeeze(), cmap="gray")
    axes[0, i].axis("off")
    axes[0, i].set_title("Real")

for i in range(8):
    axes[1, i].imshow(generated[i].reshape(28, 28), cmap="gray")
    axes[1, i].axis("off")
    axes[1, i].set_title("Generated")

plt.tight_layout()
plt.show()

In [ ]:
# 2) Write a python program to design a Restricted Boltzmann Machine for
# predicting collaboration.

# Assignment 5 - Set A - Q2
# RBM for Predicting Collaboration

import numpy as np
from sklearn.neural_network import BernoulliRBM

# Dataset: 1 = collaboration, 0 = no collaboration
X = np.array([
    [1, 1, 1, 0],
    [1, 1, 0, 1],
    [0, 1, 1, 0],
    [0, 0, 1, 1],
    [1, 0, 0, 1],
    [1, 1, 1, 1]
])

# Create RBM model
rbm = BernoulliRBM(
    n_components=2,
    learning_rate=0.1,
    n_iter=500,
    random_state=42
)

# Train model
rbm.fit(X)

# Predict collaboration pattern
test_data = np.array([[1, 1, 0, 0]])

hidden = rbm.transform(test_data)
reconstructed = rbm.gibbs(test_data)

print("Input Data:", test_data)
print("Hidden Features:", hidden)
print("Reconstructed Data:", reconstructed)

# Collaboration prediction
if reconstructed[0].mean() >= 0.5:
    print("Prediction: Collaboration Likely")
else:
    print("Prediction: Collaboration Unlikely")

In [ ]:
# 3) Implement Deep Belief network using Restricted RBM to create an effective
# stack of models to perform digit classification.

# Assignment 5 - Set A - Q3
# Deep Belief Network using RBM for Digit Classification

import numpy as np
import matplotlib.pyplot as plt
from sklearn.neural_network import BernoulliRBM
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from tensorflow.keras.datasets import mnist

# Load MNIST dataset
(X_train, y_train), (X_test, y_test) = mnist.load_data()

# Normalize and flatten images
X_train = X_train.reshape(-1, 784) / 255.0
X_test = X_test.reshape(-1, 784) / 255.0

# Use a smaller dataset for faster execution
X_train = X_train[:10000]
y_train = y_train[:10000]
X_test = X_test[:2000]
y_test = y_test[:2000]

# Convert pixel values to binary
X_train = (X_train > 0.5).astype(float)
X_test = (X_test > 0.5).astype(float)

# Create DBN using two RBM layers
dbn = Pipeline([
    ("rbm1", BernoulliRBM(
        n_components=256,
        learning_rate=0.01,
        n_iter=5,
        random_state=42
    )),
    ("rbm2", BernoulliRBM(
        n_components=64,
        learning_rate=0.01,
        n_iter=5,
        random_state=42
    )),
    ("classifier", LogisticRegression(max_iter=1000))
])

# Train the model
dbn.fit(X_train, y_train)

# Evaluate the model
accuracy = dbn.score(X_test, y_test)
print("Digit Classification Accuracy:", accuracy)

# Predict a digit
prediction = dbn.predict(X_test[:1])
print("Predicted Digit:", prediction[0])
print("Actual Digit:", y_test[0])

# Display image
plt.imshow(X_test[0].reshape(28, 28), cmap="gray")
plt.title("Predicted Digit: " + str(prediction[0]))
plt.axis("off")
plt.show()

In [ ]:
# 1) Write a python program to implement digit classification using
# autoencoders. (Use MNIST dataset)

# Assignment 6 - Set A - Q1
# Digit Classification using Autoencoders

import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.datasets import mnist
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import Input, Dense

# Load MNIST dataset
(X_train, y_train), (X_test, y_test) = mnist.load_data()

# Normalize and flatten images
X_train = X_train.reshape(-1, 784) / 255.0
X_test = X_test.reshape(-1, 784) / 255.0

# Create Autoencoder
input_layer = Input(shape=(784,))
encoded = Dense(128, activation="relu")(input_layer)
encoded = Dense(64, activation="relu")(encoded)
decoded = Dense(128, activation="relu")(encoded)
decoded = Dense(784, activation="sigmoid")(decoded)

autoencoder = Model(input_layer, decoded)

# Compile and train Autoencoder
autoencoder.compile(optimizer="adam", loss="binary_crossentropy")
autoencoder.fit(
    X_train, X_train,
    epochs=5,
    batch_size=128,
    validation_data=(X_test, X_test)
)

# Extract encoded features
encoder = Model(input_layer, autoencoder.layers[2].output)
X_train_encoded = encoder.predict(X_train, verbose=0)
X_test_encoded = encoder.predict(X_test, verbose=0)

# Create digit classifier
classifier = Sequential([
    Input(shape=(64,)),
    Dense(10, activation="softmax")
])

classifier.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# Train classifier
classifier.fit(
    X_train_encoded, y_train,
    epochs=5,
    batch_size=128,
    validation_split=0.1
)

# Evaluate classifier
loss, accuracy = classifier.evaluate(X_test_encoded, y_test, verbose=0)
print("Test Accuracy:", accuracy)
print("Test Loss:", loss)

# Predict a digit
prediction = classifier.predict(X_test_encoded[:1], verbose=0)
digit = np.argmax(prediction)

print("Predicted Digit:", digit)
print("Actual Digit:", y_test[0])

# Display image
plt.imshow(X_test[0].reshape(28, 28), cmap="gray")
plt.title("Predicted Digit: " + str(digit))
plt.axis("off")
plt.show()

In [ ]:
#2) Write a python program to implement Tic-tac-Toe using reinforcement learning.

#Ass1SetB2

import random

# --------------------------------------------------
# Q-Learning Parameters
# --------------------------------------------------
Q = {}

alpha = 0.8       # Learning rate
gamma = 0.95      # Discount factor
epsilon = 0.2     # Exploration probability


# --------------------------------------------------
# Board
# --------------------------------------------------
def print_board(board):

    print()

    for i in range(0, 9, 3):

        print(
            board[i], "|",
            board[i+1], "|",
            board[i+2]
        )

        if i < 6:
            print("--+---+--")

    print()


# --------------------------------------------------
# Get State
# --------------------------------------------------
def get_state(board):

    return "".join(board)


# --------------------------------------------------
# Available Actions
# --------------------------------------------------
def available_actions(board):

    return [
        i for i in range(9)
        if board[i] == "-"
    ]


# --------------------------------------------------
# Check Winner
# --------------------------------------------------
def check_winner(board):

    winning_positions = [
        (0,1,2),
        (3,4,5),
        (6,7,8),
        (0,3,6),
        (1,4,7),
        (2,5,8),
        (0,4,8),
        (2,4,6)
    ]

    for a, b, c in winning_positions:

        if (
            board[a] != "-"
            and board[a] == board[b]
            and board[b] == board[c]
        ):
            return board[a]

    if "-" not in board:
        return "Draw"

    return None


# --------------------------------------------------
# Choose Action
# --------------------------------------------------
def choose_action(board, player):

    state = get_state(board)

    actions = available_actions(board)

    if state not in Q:
        Q[state] = {
            action: 0.0
            for action in actions
        }

    # Exploration
    if random.random() < epsilon:

        return random.choice(actions)

    # Exploitation
    values = {
        action: Q[state].get(action, 0.0)
        for action in actions
    }

    return max(
        values,
        key=values.get
    )


# --------------------------------------------------
# Update Q-value
# --------------------------------------------------
def update_q(
    state,
    action,
    reward,
    next_state,
    next_actions
):

    if state not in Q:

        Q[state] = {}

    old_value = Q[state].get(
        action,
        0.0
    )

    if next_actions:

        max_next = max(
            Q.get(next_state, {}).get(
                a, 0.0
            )
            for a in next_actions
        )

    else:

        max_next = 0

    new_value = old_value + alpha * (
        reward +
        gamma * max_next -
        old_value
    )

    Q[state][action] = new_value


# --------------------------------------------------
# Training
# --------------------------------------------------
def train(episodes=10000):

    global Q

    for episode in range(episodes):

        board = ["-"] * 9

        player = "X"

        while True:

            state = get_state(board)

            action = choose_action(
                board,
                player
            )

            board[action] = player

            result = check_winner(board)

            # Win
            if result == player:

                reward = 1

                update_q(
                    state,
                    action,
                    reward,
                    get_state(board),
                    []
                )

                break

            # Draw
            if result == "Draw":

                reward = 0.5

                update_q(
                    state,
                    action,
                    reward,
                    get_state(board),
                    []
                )

                break

            # Switch player
            player = (
                "O"
                if player == "X"
                else "X"
            )

            next_state = get_state(board)

            next_actions = available_actions(
                board
            )

            # Small negative reward
            reward = 0

            update_q(
                state,
                action,
                reward,
                next_state,
                next_actions
            )


# --------------------------------------------------
# Train Agent
# --------------------------------------------------
print("Training Tic-Tac-Toe Agent...")

train(10000)

print("Training completed.")
print("Number of learned states:", len(Q))


# --------------------------------------------------
# Play Game
# --------------------------------------------------
def play_game():

    board = ["-"] * 9

    print("\nTic-Tac-Toe")
    print("Positions are:")
    print("0 | 1 | 2")
    print("3 | 4 | 5")
    print("6 | 7 | 8")

    print_board(board)

    while True:

        # Human = X
        move = int(
            input("Enter your move (0-8): ")
        )

        if (
            move not in available_actions(board)
        ):
            print("Invalid move!")

            continue

        board[move] = "X"

        result = check_winner(board)

        print_board(board)

        if result:

            print("Result:", result)

            break

        # AI = O
        state = get_state(board)

        actions = available_actions(board)

        if state not in Q:

            Q[state] = {
                action: 0
                for action in actions
            }

        ai_move = max(
            actions,
            key=lambda a:
            Q[state].get(a, 0)
        )

        board[ai_move] = "O"

        print("AI selected:", ai_move)

        result = check_winner(board)

        print_board(board)

        if result:

            print("Result:", result)

            break


# --------------------------------------------------
# Start Game
# --------------------------------------------------
play_game()